# Tutorial 1.12: Multi-Agent Orchestration with CrewAI + MLflow

**CrewAI** is a framework for orchestrating autonomous AI agents that collaborate as a **crew**. Each agent has a defined role, goal, and backstory, and the crew coordinates their work through either **sequential** or **hierarchical** processes.

### CrewAI vs. Other Multi-Agent Frameworks

| Feature | CrewAI | LangGraph (Tutorial 10) | Deep Agents (Tutorial 11) |
|---------|--------|------------------------|---------------------------|
| Agent definition | Role-based (role, goal, backstory) | Node functions in a graph | System prompt + tools |
| Orchestration | Sequential or hierarchical process | Explicit graph edges | Built-in planning + delegation |
| Task routing | Crew assigns tasks to agents | Supervisor node routes | Parent delegates via `task()` |
| Tool sharing | Per-agent or shared across crew | Per-node | Per-agent or inherited |
| Memory | Built-in short/long-term memory | Checkpointing | File system backend |
| Best for | Role-based collaboration | Custom control flow | Long-running autonomous work |

### What You'll Learn

1. **Example 1** — Custom tools with CrewAI: querying FEMA disaster data
2. **Example 2** — Hierarchical crew: manager delegates to specialist agents
3. **Example 3** — Evaluating crew outputs with `mlflow.genai.evaluate()`

All examples use `mlflow.crewai.autolog()` to automatically trace the full crew execution.

### Prerequisites
- Completed tutorials 01-07 (MLflow basics, tracing, evaluation)
- OpenAI API key configured in `.env`

---
## Step 1: Environment Setup

In [ ]:
# Install CrewAI (if not already installed via pyproject.toml)
%pip install -q crewai crewai-tools

In [ ]:
import os
import mlflow
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM

load_dotenv()

# Configure MLflow
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000"))
mlflow.set_experiment("12-crewai-multi-agent")

# Enable auto-tracing for CrewAI and OpenAI
mlflow.crewai.autolog()
mlflow.openai.autolog()

# Initialize the LLM
llm = LLM(model="openai/gpt-4o-mini", temperature=0.3)

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {mlflow.get_experiment_by_name('12-crewai-multi-agent').name}")
print(f"LLM: {llm.model}")

---
## Example 1: Custom Tools — FEMA Disaster Data Crew

CrewAI agents become powerful when equipped with **tools**. This example creates agents with custom tools that query the same FEMA disaster database used in Tutorial 10 (multi-agent supervisor).

![Example 1: Sequential Crew with Tools](images/12_crewai_ex1_tools.svg)

We define two tools:
- **`query_disaster_database`** — translates natural language to SQL against the FEMA SQLite database
- **`search_fema_policies`** — searches FEMA policy documents by keyword

**What to observe in MLflow traces:**
- Tool invocation spans nested inside agent execution spans
- LLM reasoning about which tool to call and with what arguments
- Tool inputs and outputs captured automatically

In [ ]:
import sqlite3
from openai import OpenAI
from crewai.tools import tool
from utils.fema_data import get_disaster_data

# Load the FEMA disaster database (200 fabricated records, 2020-2025)
disaster_data = get_disaster_data()
db_conn = sqlite3.connect(":memory:")
disaster_data.to_sql("disaster_data", db_conn, index=False, if_exists="replace")

# OpenAI client for text-to-SQL
oai_client = OpenAI()

print(f"FEMA Disaster Database: {len(disaster_data)} records")
print(f"Columns: {list(disaster_data.columns)}")
print(f"Years: {sorted(disaster_data['year'].unique())}")
print(f"Disaster types: {sorted(disaster_data['disaster_type'].unique())}")

In [ ]:
# FEMA Policy Document Corpus (same as Tutorial 10)
POLICY_DOCUMENTS = {
    "evacuation_protocols": (
        "FEMA Evacuation Protocols (ICS-300): All evacuation orders must follow the Incident Command System. "
        "Zone-based evacuation proceeds from highest-risk zones outward. Mandatory evacuation requires "
        "governor authorization. Evacuation routes must be pre-designated and communicated via Wireless "
        "Emergency Alerts (WEA). Special needs populations require dedicated transport. Shelter capacity "
        "must be verified before issuing orders. Pet-friendly shelters must be available per PETS Act."
    ),
    "federal_assistance_guidelines": (
        "FEMA Individual Assistance (IA) Program: Eligible applicants include US citizens, non-citizen nationals, "
        "and qualified aliens in presidentially declared disaster areas. Assistance types: Housing Assistance "
        "(rental, repair, replacement), Other Needs Assistance (medical, dental, funeral, transportation), "
        "and Crisis Counseling. Maximum grant per household is set annually ($42,500 for 2024). Applications "
        "must be filed within 60 days of disaster declaration. SBA disaster loans available for amounts exceeding grants."
    ),
    "flood_response_procedures": (
        "FEMA Flood Response (NRF ESF-3): Pre-event: activate flood gauges, pre-position pumps and sandbags, "
        "issue Flash Flood Watches 24-48h in advance. During event: deploy Urban Search and Rescue (US&R) "
        "teams within 6 hours, establish Points of Distribution (PODs) for water and supplies. Post-event: "
        "conduct Preliminary Damage Assessments (PDAs) within 72 hours, activate National Flood Insurance "
        "Program (NFIP) claims process. Flood zones A and V require mandatory flood insurance for federally-backed mortgages."
    ),
    "wildfire_management": (
        "FEMA Wildfire Response (NRF ESF-4): Coordinate with USFS and state forestry agencies. Pre-position "
        "resources when Fire Weather Watch is issued. Activate Fire Management Assistance Grants (FMAG) for "
        "state cost-sharing. Evacuation triggers: fire within 2 miles of populated areas with wind >25mph. "
        "Post-fire: activate Burned Area Emergency Response (BAER) teams within 7 days. Debris flow risk "
        "assessment required for all slopes >30%% in burn scar areas. Community wildfire protection plans (CWPPs) required."
    ),
    "hurricane_preparedness": (
        "FEMA Hurricane Preparedness (NRF ESF-5): 120-hour watch triggers FEMA Region activation. 72-hour "
        "warning activates Incident Management Assistance Teams (IMAT). 48 hours: pre-stage commodities "
        "(MREs, water, tarps, generators) at Federal Staging Areas. 24 hours: activate ESF-1 (Transportation) "
        "for evacuation support. Storm surge zones require mandatory evacuation for Category 3+. Post-landfall: "
        "deploy FEMA Corps teams, activate Transitional Sheltering Assistance (TSA) within 14 days."
    ),
}

print(f"Policy corpus: {len(POLICY_DOCUMENTS)} documents")

In [ ]:
SCHEMA_INFO = (
    "Table 'disaster_data' columns:\n"
    "  - disaster_id: TEXT (e.g., 'DR-4001')\n"
    "  - year: INTEGER (2020-2025)\n"
    "  - state: TEXT (e.g., 'California', 'Florida')\n"
    "  - disaster_type: TEXT ('Wildfire', 'Hurricane', 'Flood', 'Tornado', 'Earthquake')\n"
    "  - severity: INTEGER (1-5, 5=catastrophic)\n"
    "  - affected_population: INTEGER\n"
    "  - federal_aid_amount: INTEGER (in USD)\n"
    "  - declaration_date: TEXT (YYYY-MM-DD)"
)


@tool
def query_disaster_database(question: str) -> str:
    """Query the FEMA disaster database using natural language.
    Translates the question to SQL and executes it against the database.
    Use this tool for questions about disaster statistics, counts, trends,
    federal aid amounts, affected populations, and state comparisons.
    """
    # Generate SQL from natural language
    response = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    f"You are a SQL expert. Given a question, generate a SQLite SELECT query.\n"
                    f"Schema:\n{SCHEMA_INFO}\n\n"
                    f"Rules:\n- Return ONLY the SQL query, no explanation\n"
                    f"- Use only SELECT statements\n"
                    f"- Use standard SQLite functions"
                ),
            },
            {"role": "user", "content": question},
        ],
        temperature=0.0,
    )
    sql = response.choices[0].message.content.strip().strip("`").replace("sql\n", "")

    try:
        cursor = db_conn.execute(sql)
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description]
        if not rows:
            return f"Query returned no results.\nSQL: {sql}"
        result_text = " | ".join(columns) + "\n"
        for row in rows[:20]:
            result_text += " | ".join(str(v) for v in row) + "\n"
        if len(rows) > 20:
            result_text += f"... ({len(rows)} total rows)"
        return f"SQL: {sql}\n\nResults:\n{result_text}"
    except Exception as e:
        return f"SQL execution error: {e}\nGenerated SQL: {sql}"


@tool
def search_fema_policies(query: str) -> str:
    """Search FEMA policy documents for information about protocols,
    guidelines, and procedures. Use this tool for questions about
    evacuation procedures, federal assistance eligibility, flood response,
    wildfire management, or hurricane preparedness.
    """
    results = []
    query_lower = query.lower()
    for doc_id, content in POLICY_DOCUMENTS.items():
        # Simple keyword matching
        doc_keywords = doc_id.replace("_", " ").split()
        if any(kw in query_lower for kw in doc_keywords) or any(
            word in doc_id for word in query_lower.split()
        ):
            results.append(f"[{doc_id.upper()}]: {content}")
    return "\n\n".join(results) if results else f"No policy documents found for: {query}"


print("Tools defined: query_disaster_database, search_fema_policies")

In [ ]:
# Create agents with tools
data_analyst = Agent(
    role="FEMA Data Analyst",
    goal="Analyze FEMA disaster data to answer statistical questions about disasters, aid, and trends",
    backstory=(
        "You are a data analyst at FEMA specializing in disaster statistics. "
        "You use the disaster database to find counts, totals, trends, and comparisons "
        "across years, states, and disaster types. Always query the database for precise numbers."
    ),
    tools=[query_disaster_database],
    llm=llm,
    verbose=True,
)

policy_expert = Agent(
    role="FEMA Policy Expert",
    goal="Provide accurate FEMA policy guidance on disaster response protocols and procedures",
    backstory=(
        "You are a FEMA policy specialist with expertise in emergency management "
        "procedures, evacuation protocols, federal assistance programs, and disaster "
        "response guidelines. You always reference official FEMA policy documents."
    ),
    tools=[search_fema_policies],
    llm=llm,
    verbose=True,
)

print(f"Agents: {data_analyst.role}, {policy_expert.role}")

In [ ]:
# Define tasks that require tool use
data_task = Task(
    description=(
        "Analyze the FEMA disaster database to answer: "
        "How many disasters hit California between 2020 and 2024? "
        "What was the total federal aid for hurricane-related disasters in 2024? "
        "Which state had the highest severity-5 disaster count?"
    ),
    expected_output=(
        "A data report with specific numbers from the database for each question: "
        "California disaster count, hurricane aid total, and top state for severity-5 disasters."
    ),
    agent=data_analyst,
)

policy_task = Task(
    description=(
        "Based on the disaster data analysis provided, recommend the appropriate "
        "FEMA response protocols. Specifically address:\n"
        "1. What evacuation protocols apply for the most common California disaster type?\n"
        "2. What federal assistance is available for the affected populations?\n"
        "3. What are the hurricane preparedness procedures given the aid levels reported?"
    ),
    expected_output=(
        "A policy guidance document that maps each data finding to the relevant "
        "FEMA protocols and procedures, with specific policy references."
    ),
    agent=policy_expert,
)

# Run sequential crew: data analysis -> policy recommendations
fema_crew = Crew(
    agents=[data_analyst, policy_expert],
    tasks=[data_task, policy_task],
    process=Process.sequential,
    verbose=True,
)

fema_result = fema_crew.kickoff()
print("\n" + "=" * 70)
print("FEMA CREW OUTPUT")
print("=" * 70)
print(fema_result.raw)

### What to Look for in the MLflow UI

In the trace view you should see:
- **Root span**: The `Crew.kickoff()` call
- **Task spans**: One for the data analysis, one for the policy guidance
- **Tool spans**: `query_disaster_database` and `search_fema_policies` invocations with inputs/outputs
- **LLM reasoning spans**: The agent deciding which tool to call and synthesizing results
- **Sequential handoff**: Data analyst output flowing as context to the policy expert

---
## Example 2: Hierarchical Crew — FEMA Disaster Response Coordinator

A **hierarchical crew** uses a **manager agent** that autonomously delegates tasks to specialist agents. This mirrors the supervisor pattern from Tutorial 10, but using CrewAI's built-in hierarchical process instead of a custom LangGraph StateGraph.

![Example 2: Hierarchical Crew](images/12_crewai_ex2_hierarchical.svg)

The manager:
1. **Receives** the user's query
2. **Decides** which specialist(s) to engage
3. **Delegates** subtasks and collects results
4. **Synthesizes** a unified response

**What to observe in MLflow traces:**
- Manager agent span coordinating specialist agent spans
- Dynamic task delegation — the manager decides the workflow at runtime
- Deeper trace hierarchy compared to sequential crews

In [ ]:
# Create the specialist agents (reusing tools from Example 1)
hierarchical_data_analyst = Agent(
    role="Disaster Data Analyst",
    goal="Query the FEMA disaster database to provide accurate statistics and trends",
    backstory=(
        "You are a quantitative analyst at FEMA. When asked for data, you always "
        "query the disaster database for precise numbers. You present data clearly "
        "with specific counts, totals, and comparisons."
    ),
    tools=[query_disaster_database],
    llm=llm,
    verbose=True,
)

hierarchical_policy_expert = Agent(
    role="Emergency Management Policy Advisor",
    goal="Provide FEMA policy guidance and response protocol recommendations",
    backstory=(
        "You are a senior FEMA policy advisor. You reference official FEMA "
        "protocols and guidelines to recommend appropriate response procedures. "
        "You always cite specific policy frameworks (ICS, NRF, ESF)."
    ),
    tools=[search_fema_policies],
    llm=llm,
    verbose=True,
)

report_writer = Agent(
    role="FEMA Report Writer",
    goal="Produce clear, actionable reports combining data analysis and policy guidance",
    backstory=(
        "You are a FEMA communications specialist who produces briefing documents "
        "for senior leadership. You synthesize data and policy into concise, "
        "actionable reports with clear recommendations."
    ),
    llm=llm,
    verbose=True,
)

print("Specialist agents created for hierarchical crew")

In [ ]:
# Define tasks for the hierarchical crew
# In hierarchical mode, the manager decides which agent handles which task
gather_data_task = Task(
    description=(
        "Gather disaster data for the 2024 hurricane season. Find:\n"
        "- Total number of hurricane disasters declared in 2024\n"
        "- States affected and their severity levels\n"
        "- Total affected population and federal aid disbursed\n"
        "- Comparison with 2023 hurricane numbers"
    ),
    expected_output="A data summary with specific numbers for each metric.",
)

assess_policy_task = Task(
    description=(
        "Based on the 2024 hurricane data, assess which FEMA response protocols "
        "were applicable. Address:\n"
        "- Hurricane preparedness timelines that should have been activated\n"
        "- Federal assistance eligibility for affected populations\n"
        "- Any gaps between policy requirements and the disaster scale"
    ),
    expected_output="A policy assessment with specific protocol references and gap analysis.",
)

write_report_task = Task(
    description=(
        "Synthesize the data analysis and policy assessment into a briefing report. "
        "Format as:\n"
        "- **Situation Summary**: Key 2024 hurricane statistics\n"
        "- **Response Assessment**: How well protocols matched the disaster scale\n"
        "- **Recommendations**: 3 specific actions for improving hurricane preparedness"
    ),
    expected_output="A structured briefing report with situation summary, assessment, and 3 recommendations.",
)

print("Hierarchical tasks defined")

In [ ]:
# Create and run the hierarchical crew
# The manager_llm controls the manager agent that delegates work
manager_llm = LLM(model="openai/gpt-4o-mini", temperature=0.1)

hierarchical_crew = Crew(
    agents=[hierarchical_data_analyst, hierarchical_policy_expert, report_writer],
    tasks=[gather_data_task, assess_policy_task, write_report_task],
    process=Process.hierarchical,  # Manager delegates autonomously
    manager_llm=manager_llm,
    verbose=True,
)

hierarchical_result = hierarchical_crew.kickoff()
print("\n" + "=" * 70)
print("HIERARCHICAL CREW OUTPUT")
print("=" * 70)
print(hierarchical_result.raw)

### Comparing Orchestration Patterns

Compare the traces from Examples 1 and 2 in the MLflow UI:

| Aspect | Sequential (Ex 1) | Sequential + Tools (Ex 1) | Hierarchical (Ex 2) |
|--------|-------------------|--------------------------|---------------------|
| Control flow | Fixed task order | Manager decides dynamically |
| Task assignment | Pre-assigned to agents | Manager assigns at runtime |
| Trace shape | Linear chain with tool branches | Tree with manager at root |
| Flexibility | Tools add branching | High (dynamic delegation) |
| Best for | Data-dependent pipelines | Open-ended multi-expert tasks |

---
## Example 3: Evaluating CrewAI Outputs with MLflow

Just like in Tutorials 10 and 11, we use `mlflow.genai.evaluate()` to systematically assess crew outputs.

We'll evaluate using:
- **`RelevanceToQuery`** — Is the crew's response relevant to the question?
- **`Safety`** — Does the response contain harmful content?
- **`Guidelines`** — Custom rubric for FEMA response quality

In [ ]:
import pandas as pd
from mlflow.genai.scorers import RelevanceToQuery, Safety, Guidelines

# Evaluation dataset — diverse FEMA queries
eval_data = pd.DataFrame({
    "inputs": [
        {"query": "How many disasters hit California between 2020 and 2024?"},
        {"query": "What are FEMA's evacuation protocols for wildfire zones?"},
        {"query": "What was the total federal aid for hurricane disasters in 2023?"},
        {"query": "Which states had severity-5 disasters and what response procedures apply?"},
    ],
})

print(f"Evaluation dataset: {len(eval_data)} queries")
eval_data

In [ ]:
def crew_predict(inputs: dict) -> str:
    """Run a CrewAI crew to answer FEMA queries."""
    query = inputs["query"]

    # Create a focused two-agent crew for each query
    eval_data_agent = Agent(
        role="FEMA Data Analyst",
        goal="Answer questions about FEMA disaster data with precise statistics",
        backstory="You are a FEMA data analyst. Query the database for exact numbers.",
        tools=[query_disaster_database, search_fema_policies],
        llm=llm,
        verbose=False,
    )

    eval_writer_agent = Agent(
        role="Response Writer",
        goal="Produce a clear, accurate response combining data and policy information",
        backstory="You synthesize data and policy into concise answers.",
        llm=llm,
        verbose=False,
    )

    analysis_task = Task(
        description=f"Analyze the following query and gather relevant data and policy information: {query}",
        expected_output="Data findings and relevant policy information.",
        agent=eval_data_agent,
    )

    synthesis_task = Task(
        description=f"Synthesize the findings into a clear, direct answer to: {query}",
        expected_output="A concise, accurate response that directly answers the question.",
        agent=eval_writer_agent,
    )

    eval_crew = Crew(
        agents=[eval_data_agent, eval_writer_agent],
        tasks=[analysis_task, synthesis_task],
        process=Process.sequential,
        verbose=False,
    )

    result = eval_crew.kickoff()
    return result.raw


# Custom Guidelines scorer for FEMA response quality
fema_response_guidelines = Guidelines(
    name="fema_response_quality",
    guidelines=(
        "The response should: "
        "(1) Include specific numbers or data when answering statistical questions, "
        "(2) Reference specific FEMA policies or procedures when answering policy questions, "
        "(3) Be directly relevant to the question asked, "
        "(4) Provide actionable information, not vague generalizations. "
        "Responses that rely on made-up data or lack specificity should score lower."
    ),
)

print("Predict function and custom scorer defined")

In [ ]:
# Run evaluation
with mlflow.start_run(run_name="CrewAI-Evaluation"):
    eval_results = mlflow.genai.evaluate(
        data=eval_data,
        predict_fn=crew_predict,
        scorers=[
            RelevanceToQuery(),
            Safety(),
            fema_response_guidelines,
        ],
    )

print("Evaluation complete!")
eval_results.metrics

In [ ]:
# View per-row results
eval_results.tables["eval_results"]

---
## Best Practices & Key Takeaways

### CrewAI Agent Design
- **Define clear roles**: Each agent should have a specific, non-overlapping role and goal
- **Write detailed backstories**: Backstories give the LLM context for how to approach tasks — they matter more than you'd think
- **Scope tools per agent**: Give each agent only the tools relevant to their role
- **Start with sequential**: Use `Process.sequential` first; switch to `Process.hierarchical` only when you need dynamic delegation
- **Keep tasks focused**: One clear objective per task with explicit `expected_output`

### MLflow Integration
- **`mlflow.crewai.autolog()`** traces the full crew execution automatically — no manual instrumentation needed
- **Compare crew configurations**: Use MLflow experiments to A/B test sequential vs. hierarchical processes
- **Evaluate systematically**: Use `mlflow.genai.evaluate()` to assess crew outputs across diverse queries
- **Custom `Guidelines` scorers**: Tailor evaluation rubrics to your crew's specific domain (e.g., FEMA response quality)

### When to Use Which Framework

| Use CrewAI when... | Use LangGraph (Tutorial 10) when... | Use Deep Agents (Tutorial 11) when... |
|--------------------|------------------------------------|--------------------------------------|
| Role-based collaboration fits the problem | You need custom control flow (loops, conditionals) | Task is long-running and autonomous |
| Quick setup with sequential or hierarchical | You need fine-grained state management | Agent needs file system access |
| Tasks have clear handoff boundaries | You need explicit graph topology | Sub-agent delegation is central |
| Agents have distinct specializations | Performance-critical routing logic | Planning decomposition adds value |

In [ ]:
# Cleanup
db_conn.close()
print("Database connection closed.")
print("\nTutorial complete! Check the MLflow UI at http://localhost:5000")
print("Experiment: 12-crewai-multi-agent")